In [1]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

In [2]:
riders = pd.read_csv("../dataset/processed/riders_clean.csv")

print(riders.shape)
riders.head()

(45493, 20)


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken (min)
0,0xcdcd,DEHRES17DEL01,36.0,4.2,30.327968,78.046106,30.397968,78.116106,2022-02-12,21:55,22:10,Fog,Jam,2,Snack,motorcycle,3.0,No,Metropolitian,46
1,0xd987,KOCRES16DEL01,21.0,4.7,10.003064,76.307589,10.043064,76.347589,2022-02-13,14:55,15:05,Stormy,High,1,Meal,motorcycle,1.0,No,Metropolitian,23
2,0x2784,PUNERES13DEL03,23.0,4.7,18.562450,73.916619,18.652450,74.006619,2022-03-04,17:30,17:40,Sandstorms,Medium,1,Drinks,scooter,1.0,No,Metropolitian,21
3,0xc8b6,LUDHRES15DEL02,34.0,4.3,30.899584,75.809346,30.919584,75.829346,2022-02-13,09:20,09:30,Sandstorms,Low,0,Buffet,motorcycle,0.0,No,Metropolitian,20
4,0xdb64,KNPRES14DEL02,24.0,4.7,26.463504,80.372929,26.593504,80.502929,2022-02-14,19:50,20:05,Fog,Jam,1,Snack,scooter,1.0,No,Metropolitian,41


In [9]:
mask = ~riders["Time_Orderd"].astype(str).str.contains(":")

print(riders.loc[mask, "Time_Orderd"].head(20))

28     0.458333333
31     0.958333333
38     0.791666667
71           0.875
83     0.958333333
116              1
124          0.375
168    0.791666667
180    0.958333333
181          0.625
184          0.375
185    0.833333333
212    0.666666667
215    0.791666667
220           0.75
225    0.458333333
227    0.458333333
229          0.875
264    0.416666667
282    0.916666667
Name: Time_Orderd, dtype: object


In [10]:
print(riders["Time_Orderd"].sample(20))

26242          08:15
4532           12:40
6570           20:15
24677          20:25
28479          19:50
44780          17:40
5484           17:30
25598          08:55
37320          23:10
22222          20:45
29620          08:35
13816          21:55
3491           21:55
17240          22:55
37150          21:45
40264          18:25
10339          22:30
18258          17:20
36218    0.458333333
20037          20:30
Name: Time_Orderd, dtype: object


In [12]:
import pandas as pd

def parse_time(value):
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    # Normal HH:MM format
    if ":" in value:
        try:
            return pd.to_datetime(value, format="%H:%M")
        except:
            return pd.NaT

    # Excel fractional time
    try:
        f = float(value)
        if 0 <= f < 1:
            seconds = int(f * 24 * 3600)
            return pd.Timestamp("2000-01-01") + pd.Timedelta(seconds=seconds)
    except:
        pass

    return pd.NaT

In [13]:
riders["Order_Time"] = riders["Time_Orderd"].apply(parse_time)

riders["Order_Hour"] = riders["Order_Time"].dt.hour

In [14]:
mask = ~riders["Time_Orderd"].astype(str).str.contains(":")

print(riders.loc[mask, "Time_Orderd"].value_counts().head(20))

Time_Orderd
0.833333333    469
1              448
0.791666667    439
0.958333333    436
0.875          430
0.916666667    418
0.75           415
0.416666667    217
0.5            197
0.458333333    196
0.375          184
0.625           89
0.541666667     86
0.583333333     74
0.708333333     69
0.666666667     55
Name: count, dtype: int64


In [15]:
print(mask.sum())


4222


In [16]:
import pandas as pd

def normalize_time(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    # Already HH:MM
    if ":" in value:
        return value

    # Excel fractional time
    try:
        f = float(value)

        # Excel sometimes stores midnight as 1.0
        if f == 1:
            f = 0

        total_seconds = int(round(f * 24 * 60 * 60))

        hours = total_seconds // 3600
        minutes = (total_seconds % 3600) // 60

        return f"{hours:02d}:{minutes:02d}"

    except:
        return None

In [17]:
riders["Time_Orderd"] = riders["Time_Orderd"].apply(normalize_time)
riders["Time_Order_picked"] = riders["Time_Order_picked"].apply(normalize_time)

In [18]:
print(riders["Time_Orderd"].head(20))
print(riders["Time_Orderd"].sample(20))

0     21:55
1     14:55
2     17:30
3     09:20
4     19:50
5     20:25
6     14:55
7     20:30
8     20:40
9     21:15
10    20:20
11    22:30
12    08:15
13    19:30
14    12:25
15    18:35
16    20:35
17    23:20
18    21:20
19    23:35
Name: Time_Orderd, dtype: object
14484    17:30
16578    16:50
7597     23:35
35581    08:50
37646    21:45
23645    21:55
1756     19:45
30776    23:15
5316     22:15
7771     22:30
15528    21:20
20679    16:00
22753    20:20
17953    18:10
36685    22:55
38474    17:40
24148    09:50
14466    09:55
15393    08:15
12345    18:30
Name: Time_Orderd, dtype: object


In [20]:
invalid_times = riders[
    riders["Time_Orderd"].astype(str).str.startswith("24:")
]["Time_Orderd"]

print(invalid_times.value_counts())

Time_Orderd
24:05:00    18
24:10:00    13
24:15:00     8
Name: count, dtype: int64


In [21]:
def clean_time(t):
    if pd.isna(t):
        return None

    t = str(t).strip()

    # Convert Excel fractional times
    if ":" not in t:
        try:
            f = float(t)
            if f == 1:
                f = 0

            total_seconds = int(round(f * 24 * 3600))
            hours = (total_seconds // 3600) % 24
            minutes = (total_seconds % 3600) // 60

            return f"{hours:02d}:{minutes:02d}"
        except:
            return None

    # Handle values like 24:05:00
    if t.startswith("24:"):
        t = "00:" + t[3:]

    # Remove seconds if present
    parts = t.split(":")
    if len(parts) == 3:
        t = f"{parts[0]}:{parts[1]}"

    return t

In [22]:
riders["Time_Orderd"] = riders["Time_Orderd"].apply(clean_time)
riders["Time_Order_picked"] = riders["Time_Order_picked"].apply(clean_time)

In [23]:
print(riders["Time_Orderd"].sample(20))

32271    23:40
20145    23:25
8086     15:40
29187    18:15
32546    00:00
15650    09:30
4780     21:45
15187    09:10
7520     15:55
26747    18:55
9793     22:55
24852    20:50
34879    08:55
22330    19:30
30140    21:15
29363    15:55
14132    23:10
19781    19:30
27184    23:20
21941    13:25
Name: Time_Orderd, dtype: object


In [24]:
riders["Order_Hour"] = pd.to_datetime(
    riders["Time_Orderd"],
    format="%H:%M",
    errors="raise"
).dt.hour

In [25]:
print(riders["Time_Orderd"].isna().sum())
print(riders["Time_Order_picked"].isna().sum())

0
0


In [26]:
print(riders[["Time_Orderd", "Time_Order_picked"]].head())

  Time_Orderd Time_Order_picked
0       21:55             22:10
1       14:55             15:05
2       17:30             17:40
3       09:20             09:30
4       19:50             20:05


In [27]:
print(riders[["Time_Orderd", "Time_Order_picked"]].tail())

      Time_Orderd Time_Order_picked
45488       11:35             11:45
45489       19:55             20:10
45490       23:50             00:05
45491       13:35             13:40
45492       17:10             17:15


In [28]:
print(riders[riders["Time_Orderd"].str.contains("24:", na=False)].shape)
print(riders[riders["Time_Order_picked"].str.contains("24:", na=False)].shape)

(0, 22)
(0, 22)


Order Hour

In [29]:
riders["Order_Hour"] = pd.to_datetime(
    riders["Time_Orderd"],
    format="%H:%M"
).dt.hour

Pickup Hour

In [30]:
riders["Pickup_Hour"] = pd.to_datetime(
    riders["Time_Order_picked"],
    format="%H:%M"
).dt.hour

Day of Week

In [31]:
riders["Day_of_Week"] = riders["Order_Date"].dt.day_name()

Month

In [32]:
riders["Month"] = riders["Order_Date"].dt.month

Weekend

In [33]:
riders["Weekend"] = (
    riders["Order_Date"].dt.weekday >= 5
).astype(int)

Peak Period

In [34]:
def peak(hour):
    if 7 <= hour <= 10:
        return "Breakfast"
    elif 12 <= hour <= 15:
        return "Lunch"
    elif 18 <= hour <= 22:
        return "Dinner"
    else:
        return "Normal"

riders["Peak_Period"] = riders["Order_Hour"].apply(peak)

In [35]:
print(riders[[
    "Time_Orderd",
    "Order_Hour",
    "Pickup_Hour",
    "Day_of_Week",
    "Weekend",
    "Peak_Period"
]].head(10))

  Time_Orderd  Order_Hour  Pickup_Hour Day_of_Week  Weekend Peak_Period
0       21:55          21           22    Saturday        1      Dinner
1       14:55          14           15      Sunday        1       Lunch
2       17:30          17           17      Friday        0      Normal
3       09:20           9            9      Sunday        1   Breakfast
4       19:50          19           20      Monday        0      Dinner
5       20:25          20           20    Saturday        1      Dinner
6       14:55          14           15     Tuesday        0       Lunch
7       20:30          20           20   Wednesday        0      Dinner
8       20:40          20           20      Sunday        1      Dinner
9       21:15          21           21     Tuesday        0      Dinner


In [36]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth's radius in kilometers

    lat1, lon1 = radians(lat1), radians(lon1)
    lat2, lon2 = radians(lat2), radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (sin(dlat / 2) ** 2 +
         cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2)

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c

In [37]:
riders["Trip_Distance_km"] = riders.apply(
    lambda row: haversine(
        row["Restaurant_latitude"],
        row["Restaurant_longitude"],
        row["Delivery_location_latitude"],
        row["Delivery_location_longitude"]
    ),
    axis=1
)

In [38]:
print(riders["Trip_Distance_km"].describe())

print(riders["Trip_Distance_km"].head())

print(riders["Trip_Distance_km"].isnull().sum())

count    45493.000000
mean        88.595073
std       1013.510526
min          1.465067
25%          4.663456
50%          9.263232
75%         13.763896
max      19688.001288
Name: Trip_Distance_km, dtype: float64
0    10.280582
1     6.242319
2    13.787860
3     2.930258
4    19.396618
Name: Trip_Distance_km, dtype: float64
0


In [39]:
print(riders["Trip_Distance_km"].describe())

count    45493.000000
mean        88.595073
std       1013.510526
min          1.465067
25%          4.663456
50%          9.263232
75%         13.763896
max      19688.001288
Name: Trip_Distance_km, dtype: float64


In [40]:
coord_cols = [
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude"
]

print(riders[coord_cols].describe())

       Restaurant_latitude  Restaurant_longitude  Delivery_location_latitude  \
count         45493.000000          45493.000000                45493.000000   
mean             17.046519             70.324924                   17.468729   
std               8.132016             22.593659                    7.334498   
min             -30.905562            -88.352885                    0.010000   
25%              12.933284             73.170283                   12.988453   
50%              18.551440             75.898497                   18.634382   
75%              22.728163             78.044095                   22.785049   
max              30.914057             88.433452                   31.054057   

       Delivery_location_longitude  
count                 45493.000000  
mean                     70.847174  
std                      21.113075  
min                       0.010000  
25%                      73.280000  
50%                      76.002471  
75%                 

In [41]:
for col in coord_cols:
    print(f"\n{col}")
    print("Min:", riders[col].min())
    print("Max:", riders[col].max())


Restaurant_latitude
Min: -30.905562
Max: 30.914057

Restaurant_longitude
Min: -88.352885
Max: 88.433452

Delivery_location_latitude
Min: 0.01
Max: 31.054057

Delivery_location_longitude
Min: 0.01
Max: 88.563452


In [42]:
outliers = riders[riders["Trip_Distance_km"] > 100]

print("Number of trips >100 km:", len(outliers))

outliers[[
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude",
    "Trip_Distance_km"
]].head(20)

Number of trips >100 km: 399


,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Trip_Distance_km
83,-27.163303,78.057044,27.233303,78.127044,6048.631240
274,-27.165108,78.015053,27.225108,78.075053,6047.919478
280,-18.551440,-73.804855,18.611440,73.864855,16612.239055
416,-30.366322,-78.070453,30.496322,78.200453,17744.205087
525,-12.325461,-76.632278,12.385461,76.692278,17118.828366
574,-12.316967,76.603067,12.326967,76.613067,2740.280655
753,-23.359194,-85.325447,23.449194,85.415447,19070.337839
1050,-26.477750,-80.351569,26.487750,80.361569,18097.339322
1082,-15.546594,73.760431,15.606594,73.820431,3464.082720
1148,-12.337978,76.616792,12.387978,76.666792,2749.406397


In [43]:
coord_cols = [
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude"
]

for col in coord_cols:
    riders[col] = riders[col].abs()

In [44]:
riders["Trip_Distance_km"] = riders.apply(
    lambda row: haversine(
        row["Restaurant_latitude"],
        row["Restaurant_longitude"],
        row["Delivery_location_latitude"],
        row["Delivery_location_longitude"]
    ),
    axis=1
)

In [45]:
print(riders["Trip_Distance_km"].describe())

count    45493.000000
mean         9.734510
std          5.607967
min          1.465067
25%          4.663345
50%          9.220148
75%         13.681492
max         20.969489
Name: Trip_Distance_km, dtype: float64


In [46]:
print(riders[riders["Trip_Distance_km"] > 50].shape)

(0, 28)


In [47]:
print(riders["Trip_Distance_km"].describe())

count    45493.000000
mean         9.734510
std          5.607967
min          1.465067
25%          4.663345
50%          9.220148
75%         13.681492
max         20.969489
Name: Trip_Distance_km, dtype: float64


Traffic Features

In [48]:
traffic_map = {
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Jam": 4
}

riders["Traffic_Score"] = (
    riders["Road_traffic_density"]
    .map(traffic_map)
)

print(riders[["Road_traffic_density", "Traffic_Score"]].head())
print(riders["Traffic_Score"].value_counts())

  Road_traffic_density  Traffic_Score
0                  Jam              4
1                 High              3
2               Medium              2
3                  Low              1
4                  Jam              4
Traffic_Score
1    15986
4    14139
2    10945
3     4423
Name: count, dtype: int64


Weather Features

In [49]:
weather_map = {
    "Sunny": 1,
    "Cloudy": 2,
    "Windy": 3,
    "Fog": 4,
    "Stormy": 5,
    "Sandstorms": 6
}

riders["Weather_Score"] = (
    riders["Weather_conditions"]
    .map(weather_map)
)

print(riders[["Weather_conditions", "Weather_Score"]].head())
print(riders["Weather_Score"].value_counts())

  Weather_conditions  Weather_Score
0                Fog              4
1             Stormy              5
2         Sandstorms              6
3         Sandstorms              6
4                Fog              4
Weather_Score
4    8178
5    7584
2    7533
6    7494
3    7422
1    7282
Name: count, dtype: int64


Vehicle Features

In [50]:
vehicle_map = {
    "bicycle": 1,
    "electric_scooter": 2,
    "scooter": 3,
    "motorcycle": 4
}

riders["Vehicle_Score"] = (
    riders["Type_of_vehicle"]
    .map(vehicle_map)
)

print(riders[["Type_of_vehicle", "Vehicle_Score"]].head())
print(riders["Vehicle_Score"].value_counts())

  Type_of_vehicle  Vehicle_Score
0      motorcycle              4
1      motorcycle              4
2         scooter              3
3      motorcycle              4
4         scooter              3
Vehicle_Score
4    26421
3    15241
2     3778
1       53
Name: count, dtype: int64


Rider Feature 

In [51]:
riders["Rider_Experience"] = (
    riders["Delivery_person_Age"] *
    riders["Delivery_person_Ratings"]
)

riders["Workload"] = riders["multiple_deliveries"]

print(riders[[
    "Delivery_person_Age",
    "Delivery_person_Ratings",
    "Rider_Experience",
    "Workload"
]].head())

   Delivery_person_Age  Delivery_person_Ratings  Rider_Experience  Workload
0                 36.0                      4.2             151.2       3.0
1                 21.0                      4.7              98.7       1.0
2                 23.0                      4.7             108.1       1.0
3                 34.0                      4.3             146.2       0.0
4                 24.0                      4.7             112.8       1.0


In [ ]:
riders.to_csv(
    "../dataset/processed/riders_features.csv",
    index=False
)

Feature Engineering Completed Successfully!
